# 05. SQL Set Operations: Union, Intersect & Except: Beginner Guide

### 📝 SQL Execution Order for Set Operations:
```text
┌─ Execution Order ────────────────────────────────────────────────────────────┐
│ 1. QUERY 1 & 2 (Execute Branches) ➔ 2. STACK (UNION / INTERSECT / EXCEPT)    │
│ ➔ 3. DEDUPLICATE (Unique Pass) ➔ 4. ORDER BY & LIMIT (Global Sort & Page)    │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

### 📌 Overview & Architectural Context
Welcome to **05. SQL Set Operations: Union, Intersect & Except**. While table joins combine relations horizontally by expanding column attributes, set operations combine relations vertically by stacking rows from independent query streams. This notebook covers vertical compatibility rules (column counts and data types), duplicate-removing union (`UNION`), raw performance stacking (`UNION ALL`), common intersection (`INTERSECT`), and set difference extraction (`EXCEPT` / `MINUS`).

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Vertical Compatibility Rules: Schema Alignment & Column Types
- [x] 🔹 Deduplicated Stacking: `UNION`
- [x] 🔹 High-Performance Raw Stacking: `UNION ALL`
- [x] 🔹 Shared Tuple Extraction: `INTERSECT`
- [x] 🔹 Set Difference Filtering: `EXCEPT` (`MINUS`)
- [x] 🔍 Scenario: Unified Multi-Channel Customer Ingestion Audit Log









In [1]:
# Setup in-memory SQLite relational engine with Native SQL Studio Execution
import sqlite3
import pandas as pd
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

conn = sqlite3.connect(':memory:')

def load_table(name, path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.to_sql(name, conn, index=False, if_exists='replace')

load_table('transactions', 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv')
load_table('customers', 'data/customers.csv' if os.path.exists('data/customers.csv') else '../data/customers.csv')
load_table('merchants', 'data/merchants.csv' if os.path.exists('data/merchants.csv') else '../data/merchants.csv')
load_table('disputes', 'data/disputes.csv' if os.path.exists('data/disputes.csv') else '../data/disputes.csv')

def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK', 'SAVEPOINT')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# Register automatic raw SQL transformer & %%sql magic
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("SQL Studio Environment Active! You can now write and run pure SQL queries directly.")


SQL Studio Environment Active! You can now write and run pure SQL queries directly.


### 🔹 High-Performance Stacking: `UNION ALL`
- **What it does:** Concatenates output streams from multiple queries vertically without checking for or removing duplicate rows.
- **Syntax:** `SELECT cols FROM table_1 UNION ALL SELECT cols FROM table_2`
- **Dataset Application & Code Demonstration:** Combines domestic and international transaction streams.


In [2]:
%%sql
SELECT transaction_id, customer_id, transaction_amount, 'High-Value' AS classification
FROM transactions
WHERE transaction_amount > 1800.00

UNION ALL

SELECT transaction_id, customer_id, transaction_amount, 'Fraud-Flagged' AS classification
FROM transactions
WHERE is_fraud = 1
LIMIT 6;


,transaction_id,customer_id,transaction_amount,classification
0,TX106376,C76616,1819.11,High-Value
1,TX111101,C41830,1998.80,High-Value
2,TX107785,C31178,1806.44,High-Value
3,TX109810,C25396,1916.91,High-Value
4,TX113708,C12719,1911.12,High-Value
5,TX110097,C82309,1917.88,High-Value


### 🔹 Deduplicated Vertical Stacking: `UNION`
- **What it does:** Combines multiple result streams into a single relation while executing a distinct pass to strip identical duplicate tuples.
- **Syntax:** `SELECT cols FROM table_1 UNION SELECT cols FROM table_2`
- **Dataset Application & Code Demonstration:** Extracts a deduplicated roster of all active entity IDs across customers and merchants.


In [3]:
%%sql
SELECT customer_id AS entity_id, 'CUSTOMER' AS entity_type FROM transactions WHERE customer_id IS NOT NULL
UNION
SELECT merchant_id AS entity_id, 'MERCHANT' AS entity_type FROM transactions WHERE merchant_id IS NOT NULL
LIMIT 6;


,entity_id,entity_type
0,C10053,CUSTOMER
1,C10074,CUSTOMER
2,C10150,CUSTOMER
3,C10221,CUSTOMER
4,C10282,CUSTOMER
5,C10336,CUSTOMER


### 🔹 Shared Record Intersection: `INTERSECT`
- **What it does:** Evaluates two independent query streams and outputs only tuples that exist in *both* result sets.
- **Syntax:** `SELECT cols FROM query_1 INTERSECT SELECT cols FROM query_2`
- **Dataset Application & Code Demonstration:** Identifies customer IDs that conducted transactions in both North and South regions.


In [4]:
%%sql
SELECT customer_id FROM transactions WHERE region = 'North'
INTERSECT
SELECT customer_id FROM transactions WHERE region = 'South'
LIMIT 5;


,customer_id
0,C10053
1,C10074
2,C10221
3,C10282
4,C10336


### 🔹 Set Difference Extraction: `EXCEPT` (`MINUS`)
- **What it does:** Returns rows from the first query stream that do *not* appear in the second query stream.
- **Syntax:** `SELECT cols FROM query_1 EXCEPT SELECT cols FROM query_2`
- **Dataset Application & Code Demonstration:** Finds customers who transacted via Visa but never transacted via Amex.


In [5]:
%%sql
SELECT customer_id FROM transactions WHERE card_type = 'Visa'
EXCEPT
SELECT customer_id FROM transactions WHERE card_type = 'Amex'
LIMIT 5;


,customer_id
0,C11934
1,C12540
2,C12560
3,C12757
4,C14117


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Set Theory vs Relational Joins Performance Comparison
- **Objective:** Compare computational complexity of `EXCEPT` vs `LEFT JOIN ... WHERE right.key IS NULL`.
- **Approach:** Demonstrate how relational optimizers transform set operations into hash anti-joins.


In [6]:
%%sql
SELECT customer_id, 'Target Segment' AS segment_tag
FROM transactions
WHERE transaction_amount > 1500.00
EXCEPT
SELECT customer_id, 'Target Segment' AS segment_tag
FROM transactions
WHERE is_fraud = 1
LIMIT 5;


,customer_id,segment_tag
0,C10150,Target Segment
1,C10371,Target Segment
2,C11824,Target Segment
3,C11854,Target Segment
4,C12757,Target Segment
